# Checkpoint 5: Query-Based Detector (RT-DETR-L)

Fine-tunes COCO-pretrained RT-DETR-L on VisDrone and evaluates it with the shared COCO protocol
(sequence-held-out validation split, every 3rd frame, ignored regions treated as "don't care").

Sections:
1. Setup
2. Configuration
3. Data
4. Training (resumes automatically from the last saved epoch)
5. Training curves
6. Validation metrics
7. Kaggle submission

## 1. Setup

Set `RUNTIME` to `"colab"` (checkpoints go to Google Drive) or `"local"` (checkpoints go to `runs/`).
On Colab, upload `kaggle.json` to the session first so the dataset can be downloaded.

In [ ]:
RUNTIME = "local"  # "colab" or "local"
REPO_URL = "https://github.com/rudkoLA/ai-hw-2-rudko-omelianchuk.git"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if RUNTIME == "colab":
    repo_dir = Path("/content/ai-hw-2-rudko-omelianchuk")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    uploaded_key = Path("/content/kaggle.json")
    if uploaded_key.exists():
        kaggle_key = Path.home() / ".kaggle" / "kaggle.json"
        kaggle_key.parent.mkdir(parents=True, exist_ok=True)
        kaggle_key.write_bytes(uploaded_key.read_bytes())
        kaggle_key.chmod(0o600)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pycocotools", "kaggle"], check=True)

print(f"Working directory: {Path.cwd()}")

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from pycocotools.coco import COCO
from ultralytics import RTDETR

from src.evaluation import evaluate_coco_metrics
from src.rtdetr import build_rtdetr_cfg, make_rtdetr_trainer
from src.setup_utils import ensure_dataset, resolve_storage, setup_environment
from src.submission import build_submission, load_test_frames, submit_to_kaggle, validate_submission
from src.ultralytics_utils import predict_coco, training_state, write_yolo_split

env = setup_environment(seed=42, mount_gdrive=False)
dataset_dir, coco_train, coco_val, class_names = ensure_dataset()
storage = resolve_storage(RUNTIME, "cp5_rtdetr")

## 2. Configuration

All tunable parameters. Give every experiment its own `run_name`: an existing run with the same name
is resumed (or skipped if finished) instead of being retrained.
Any other Ultralytics `train()` argument can be added to `TRAIN_EXTRA`.

In [ ]:
CFG = {
    "run_name": "rtdetr_l_640_q300",
    "weights": "rtdetr-l.pt",       # COCO-pretrained
    "frame_stride": 3,
    # Schedule
    "imgsz": 640,
    "epochs": 30,
    "batch": 8,
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "backbone_lr_mult": 0.1,         # backbone LR = lr0 * backbone_lr_mult
    "weight_decay": 1e-4,
    "warmup_epochs": 1,
    "warmup_bias_lr": 0.0,
    # Queries and denoising
    "num_queries": 300,
    "num_denoising": 100,            # 0 disables denoising training
    # Hungarian matching costs and loss weights
    "matcher_cost": {"class": 2.0, "bbox": 5.0, "giou": 2.0},
    "loss_gain": {"class": 1.0, "bbox": 5.0, "giou": 2.0},
    # Misc
    "workers": 4,
    "seed": 42,
}

TRAIN_EXTRA = {}  # e.g. {"mosaic": 0.5, "close_mosaic": 5, "patience": 10}

## 3. Data

Writes Ultralytics image lists for every `frame_stride`-th frame of each sequence. The frame selection is
the same one used by `VisDroneDataset`, so all detectors are trained and evaluated on identical frames.

In [ ]:
data_yaml, frames = write_yolo_split(
    dataset_dir,
    {"train": coco_train, "val": coco_val},
    out_dir=storage / "lists",
    class_names=class_names,
    stride=CFG["frame_stride"],
)
del coco_train  # large; only the selected frames are needed from here on

## 4. Training

Checkpoints (`last.pt` every epoch, `best.pt`) and `results.csv` are written to `storage/<run_name>`.
Re-running this cell after an interruption resumes from `last.pt`; a finished run is not retrained.

In [ ]:
model_cfg = build_rtdetr_cfg("rtdetr-l.yaml", num_queries=CFG["num_queries"], num_denoising=CFG["num_denoising"])
Trainer = make_rtdetr_trainer(
    model_cfg,
    matcher_cost=CFG["matcher_cost"],
    loss_gain=CFG["loss_gain"],
    backbone_lr_mult=CFG["backbone_lr_mult"],
)

run_dir = storage / CFG["run_name"]
state = training_state(run_dir)
print(f"Run '{CFG['run_name']}': {state}")

if state == "new":
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "cfg.json").write_text(json.dumps({**CFG, "train_extra": TRAIN_EXTRA}, indent=2))
    RTDETR(CFG["weights"]).train(
        trainer=Trainer,
        data=str(data_yaml),
        project=str(storage),
        name=CFG["run_name"],
        exist_ok=True,
        imgsz=CFG["imgsz"],
        epochs=CFG["epochs"],
        batch=CFG["batch"],
        optimizer=CFG["optimizer"],
        lr0=CFG["lr0"],
        weight_decay=CFG["weight_decay"],
        warmup_epochs=CFG["warmup_epochs"],
        warmup_bias_lr=CFG["warmup_bias_lr"],
        workers=CFG["workers"],
        seed=CFG["seed"],
        **TRAIN_EXTRA,
    )
elif state == "resume":
    RTDETR(str(run_dir / "weights" / "last.pt")).train(trainer=Trainer, resume=True)

## 5. Training curves

Per-epoch validation mAP and losses logged by Ultralytics. These per-epoch metrics use Ultralytics' own
evaluator; the final numbers in Section 6 come from the shared COCO evaluation.

In [ ]:
history = pd.read_csv(run_dir / "results.csv")
history.columns = history.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for col, label in [("metrics/mAP50-95(B)", "mAP@[.5:.95]"), ("metrics/mAP50(B)", "mAP@.5")]:
    axes[0].plot(history["epoch"], history[col], marker="o", label=label)
axes[0].set(title="Validation mAP", xlabel="epoch")
for ax, split in zip(axes[1:], ["train", "val"]):
    for col in [c for c in history.columns if c.startswith(f"{split}/")]:
        ax.plot(history["epoch"], history[col], label=col.split("/")[1])
    ax.set(title=f"{split.capitalize()} losses", xlabel="epoch")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Validation metrics

`best.pt` is evaluated on the validation frames with `evaluate_coco_metrics` (up to 500 detections per image;
RT-DETR returns at most `num_queries` boxes and needs no NMS). Results are saved to `storage/<run_name>/metrics.json`.

In [ ]:
best_model = RTDETR(str(run_dir / "weights" / "best.pt"))
records, ms_per_frame = predict_coco(
    best_model,
    frames["val"],
    image_root=dataset_dir / "val",
    imgsz=CFG["imgsz"],
    conf=0.001,
    max_det=CFG["num_queries"],
    batch=CFG["batch"],
)

coco_gt = COCO(str(dataset_dir / "val" / "annotations_coco.json"))
metrics = evaluate_coco_metrics(coco_gt, records, image_ids=[img["id"] for img in frames["val"]], max_dets=500)
metrics.update(
    {
        "ms_per_frame": ms_per_frame,
        "fps": 1000.0 / ms_per_frame,
        "params_M": sum(p.numel() for p in best_model.model.parameters()) / 1e6,
        "epochs_trained": int(history["epoch"].max()),
    }
)
(run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

pd.Series(metrics, name=CFG["run_name"]).round(4).to_frame()

## 7. Kaggle submission

Predicts all test frames with `best.pt` and writes `storage/<run_name>/submission.csv`
(boxes in original pixel coordinates, at most 500 detections per image, empty rows for frames without detections).

In [ ]:
test_frames = load_test_frames(dataset_dir)
test_records, _ = predict_coco(
    best_model,
    test_frames,
    image_root=dataset_dir,
    imgsz=CFG["imgsz"],
    conf=0.001,
    max_det=CFG["num_queries"],
    batch=CFG["batch"],
)

submission_path = run_dir / "submission.csv"
build_submission(test_records, test_frames, submission_path)
report = validate_submission(submission_path, test_frames)
report

Set `SUBMIT = True` to upload the file (requires `~/.kaggle/kaggle.json`).

In [ ]:
SUBMIT = False
SUBMISSION_MESSAGE = f"CP5 RT-DETR-L {CFG['run_name']}"

if SUBMIT:
    assert report["is_valid"], report["errors"]
    submit_to_kaggle(submission_path, SUBMISSION_MESSAGE)